#   Text clasification para Kaggle

In [1]:
# Data processing
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import copy

import time
import datetime

from sklearn.metrics import confusion_matrix, cohen_kappa_score

from datasets import Dataset,  DatasetDict

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

# Modeling
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding, AutoModelForSequenceClassification, AdamW, get_scheduler

# Progress bar
from tqdm.auto import tqdm

from utils import plot_confusion_matrix, get_artifact_filename

from joblib import load, dump

# Verificamos que CUDA está funcional
torch.cuda.is_available()

c:\Users\josek\miniconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

**Bajamos el modelo**

In [2]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

c:\Users\josek\miniconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


**Armado de los Datasets**

In [3]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train_FE.csv")
PATH_TO_TEST = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/test/test.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

# Parametros y variables
SEED = 42
TEST_SIZE = 0.2

BATCH_SIZE = 15

MODEL_NAME = '05 Kaggle'

MODEL_VERSION = '1.1'

# MODEL_NAME = '06 Bert'

# MODEL_VERSION = '1.1'

In [4]:
# Cargar los datos
df = pd.read_csv(PATH_TO_TRAIN)
df = df[df['Description'].notnull()]
df['labels'] = df["AdoptionSpeed"]

# Dividir los datos usando sklearn
#train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df.AdoptionSpeed)

study_lgb = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name="04 - LGB Multiclass CV - FE12",
                           load_if_exists = True)

lgb_test_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

train_df = df
# test_df = df[df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)

# Convertir a Dataset
train_dataset = Dataset.from_pandas(train_df)

# Combinar en un DatasetDict
dataset = DatasetDict({
    'train': train_dataset
})

# Codificar la columna de etiquetas como clases
dataset = dataset.class_encode_column('labels')

# Hacer una lista de columnas para remover antes de la tokenización
cols_to_remove = [col for col in dataset["train"].column_names if col != 'labels']
print(cols_to_remove)

[I 2025-09-01 20:43:04,674] Using an existing study with name '04 - LGB Multiclass CV - FE12' instead of creating a new one.
Casting to class labels: 100%|██████████| 14980/14980 [00:00<00:00, 289599.66 examples/s]

['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', 'HasName', 'DescLen', 'score_salud', 'Puro', 'AgeRange', 'Color_comb', 'Content', 'logFee', 'polarity', 'Subjectivity', 'state_gdp', 'state_population', 'gdp_vs_population', '__index_level_0__']


In [5]:
lgb_test_dataset.head().PetID

14696    8f20e24ef
14823    2d72ef0c4
2838     44cd12263
1848     210c4a637
669      21493e6ea
Name: PetID, dtype: object

In [6]:
# Tokenize and encode the dataset
def tokenize(batch):
    from transformers import DistilBertTokenizerFast
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    tokenized_batch = tokenizer(batch["Description"], padding=True, truncation=True, max_length=512)
    return tokenized_batch

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=4)

# Set dataset format for PyTorch
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Check the output
print(dataset_enc["train"].column_names)
     


Map (num_proc=4): 100%|██████████| 14980/14980 [00:06<00:00, 2413.95 examples/s]

['labels', 'input_ids', 'attention_mask']


In [7]:
# Instantiate a data collator with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create data loaders for to reshape data for PyTorch model
train_dataloader = DataLoader(
    dataset_enc["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)

In [8]:
# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", 
                                                           num_labels=num_labels)

Number of labels: 5


c:\Users\josek\miniconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:

# Set the device automatically (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Move model to device
model.to(device)

cuda


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [10]:
def train_model(model, dataloaders, device, num_epochs=4, lr=0.001, trial=None):
    
    since = time.time()

    # Create the optimizer
    optimizer = AdamW(model.parameters(), lr=lr)

    # Further define learning rate scheduler
    num_training_batches = len(train_dataloader)
    num_training_steps = num_epochs * num_training_batches
    lr_scheduler = get_scheduler(
        "linear",                   # linear decay
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )


    # best_model_wts = copy.deepcopy(model.state_dict())
    # best_acc = 0.0
    # best_kappa =  -999

    # train_losses = []
    # val_losses = []

    # try:
    #     previous_best = study.best_value
    # except:
    #     previous_best = -999

    phase = 'train'
    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)
        
        # kappa_labels_true = []
        # kappa_labels_predicted = []
        # output_scores = []

        
        
        if phase == 'train':
            model.train()  # Set model to training mode
        else:
            model.eval()   # Set model to evaluate mode

        running_loss = 0.0
        running_corrects = 0

        # Iterate over data.
        for batch in tqdm(dataloaders[phase]):
            batch = batch.to(device)
            #inputs = inputs.to(device)
            labels = batch.labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward
            # Track history if only in train
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(**batch)
                loss = outputs.loss

                preds = torch.nn.functional.softmax(outputs.logits, dim=-1)
                preds_labels = torch.argmax(preds, dim=-1)


                # Backward + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Statistics
            running_loss += loss.item() * labels.size(0)
            running_corrects += torch.sum(preds_labels == labels.data)
            
            #END OF BATCH

        # epoch_loss = running_loss / len(datasets[phase])
        # epoch_acc = running_corrects.double() / len(datasets[phase])
        
        # if phase == 'train':
        #     train_losses.append(epoch_loss)
        #     kappa_score = np.nan
                


        # print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {kappa_score:.3f}')

        #END OF PHASE

        #END OF EPOCH

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    # print('Best val Acc: {:.2f}%'.format(best_acc * 100))

    return model



In [11]:
# MODEL_NAME = '06 Bert'

# MODEL_VERSION = '1.1'

MODEL_NAME_PARAM = '06 Bert'

MODEL_VERSION_PARAM = '1.3'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME_PARAM}_{MODEL_VERSION_PARAM}',
                            load_if_exists = True)

[I 2025-09-01 20:43:47,682] Using an existing study with name '06 Bert_1.3' instead of creating a new one.


In [12]:
epochs = study_bert.best_params["epochs"]
lr = study_bert.best_params["lr"]

In [14]:

# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")


Number of labels: 5


In [15]:

best_model = train_model(model,
                       dataloaders={'train': train_dataloader},
                       device=device, 
                       lr = lr,
                       num_epochs=epochs)


c:\Users\josek\miniconda3\envs\ldi2_cuda\Lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0/3
----------


100%|██████████| 999/999 [07:23<00:00,  2.25it/s]


Epoch 1/3
----------


100%|██████████| 999/999 [07:15<00:00,  2.30it/s]


Epoch 2/3
----------


100%|██████████| 999/999 [07:29<00:00,  2.22it/s]


Epoch 3/3
----------


100%|██████████| 999/999 [07:59<00:00,  2.09it/s]

Training complete in 30m 8s


In [16]:
# Guardo el modelo
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
print(f'Modelo guardado en {model_path}')

Modelo guardado en ../work/optuna_temp_artifacts\05 Kaggle_1.1_20250901_211410.pth


In [17]:
best_model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [18]:
best_model.num_labels

5

In [19]:
df = pd.read_csv(PATH_TO_TEST)
df = df[df['Description'].notnull()]

In [20]:
import gc
gc.collect()
torch.cuda.empty_cache()



In [21]:
print(torch.cuda.get_device_name(0))
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Memoria en uso: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"Memoria reservada: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.00 GB
Memoria en uso: 0.52 GB
Memoria reservada: 0.79 GB


In [22]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Dataset wrapper
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {key: val.squeeze(0) for key, val in enc.items()}

# Parámetros
batch_size = 16  # podés ajustar según memoria
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)
best_model.eval()

# Crear DataLoader
dataset = TextDataset(df["Description"].tolist(), tokenizer)
loader = DataLoader(dataset, batch_size=batch_size)

# Predicciones por lotes
all_probs = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = best_model(**batch)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu())

# Concatenar
probabilities = torch.cat(all_probs, dim=0).numpy()

# Guardar en el DataFrame
df["pred"] = list(probabilities)




In [23]:
df

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,pred
0,2,Dopey & Grey,8,266,266,1,2,6,7,1,...,1,2,0,41326,2ece3b2573dcdcebd774e635dca15fd9,0,"Dopey Age: 8mths old Male One half of a pair, ...",e2dfc2935,2.0,"[0.003846097, 0.27826887, 0.6002159, 0.1147330..."
1,2,Chi Chi,36,285,264,2,1,4,7,2,...,2,1,0,41326,2ece3b2573dcdcebd774e635dca15fd9,0,"Please note that Chichi has been neutered, the...",f153b465f,1.0,"[0.05020208, 0.35483414, 0.32617506, 0.2250589..."
2,2,Sticky,2,265,0,1,6,7,0,2,...,1,1,200,41326,e59c106e9912fa30c898976278c2e834,0,"Sticky, named such because of his tendency to ...",3c90f3f54,4.0,"[0.021796627, 0.7062811, 0.21454561, 0.0510749..."
3,1,Dannie & Kass [In Penang],12,307,0,2,2,5,0,2,...,1,2,0,41326,e59c106e9912fa30c898976278c2e834,0,Dannie and Kass are mother and daughter. We en...,e02abc8a3,5.0,"[0.0011437818, 0.005607937, 0.008500331, 0.009..."
4,2,Cuddles,12,265,0,1,2,3,7,2,...,1,1,0,41326,e59c106e9912fa30c898976278c2e834,0,"Extremely cuddly cat, hence the origin of his ...",09f0df7d1,5.0,"[0.009793902, 0.32857242, 0.33875132, 0.231386..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3967,1,Hugo,5,307,307,1,1,2,0,2,...,1,2,150,41401,18ec8ca4486bc2760de0bd5390cee30c,0,Found on the streets. Treated for mange. They ...,ae57f8d52,5.0,"[0.016190462, 0.12927191, 0.11473601, 0.140699..."
3968,1,Spot,30,307,307,1,1,2,7,2,...,1,1,0,41326,d83be5f5e2d04e24d7376e99eafd8708,0,Very good guard dog. Healthy was found in Fron...,83432904d,2.0,"[0.0075623663, 0.15285726, 0.59768724, 0.19885..."
3969,2,NaN,5,300,0,3,1,2,4,2,...,1,6,0,41401,30aa45fdbe45e39d5614ef583b569073,0,these cat's mother was killed when they was ne...,399013029,1.0,"[0.0084952405, 0.029026678, 0.037987005, 0.044..."
3970,1,Smokey,24,307,0,2,5,7,0,2,...,1,1,0,41325,087903c2819a6297519c93d962b488d5,0,"smokey is good family pet. very obedient,so lo...",fd80b8c80,3.0,"[0.0054038693, 0.013785562, 0.19734251, 0.4975..."


In [24]:
df.to_csv("bert_kaggle_pred.csv", index= False)